# SFIL Text2SQL SFT Pipeline (Adapted)

"
"This notebook is adapted to the project pipeline in this repository:
"
"- generation (`script 03`)
"
"- deterministic validation (`script 04`)
"
"- LLM judge (`script 05/06`)

"
"It builds a fine-tuning dataset from per-model judge outputs (`data/intermediate/by_model/*`) and provides an optional Unsloth + LoRA training flow for Qwen-style models.


In [ ]:
# Imports and root resolution
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict


def resolve_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "assets").exists() and (cur / "data").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Could not resolve repository root from current directory.")


ROOT = resolve_repo_root(Path.cwd())
print("ROOT =", ROOT)


In [ ]:
# Configuration
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BY_MODEL_DIR = ROOT / "data" / "intermediate" / "by_model"
ASSETS_DIR = ROOT / "assets"
OUTPUT_DIR = ROOT / "data" / "finetune"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ALIASES = [
    "deepseek_r1",
    "gpt_oss_120b",
    "mistral_small_24b",
    "qwen_7b",
]

SOURCE_KIND = "pass"  # "pass" (recommended) or "judged"
PASS_SCORE_THRESHOLD = 4.0

SCHEMA_SQL_PATH = ASSETS_DIR / "schema_sqlite.sql"
SCHEMA_GUIDE_PATH = ASSETS_DIR / "schema_guide.yaml"
BUSINESS_NARRATIVE_PATH = ASSETS_DIR / "business_context_sfil.md"

print("Using aliases:", MODEL_ALIASES)
print("Source kind:", SOURCE_KIND)


In [ ]:
# Load prompt context from assets
def read_text(path: Path, default: str = "") -> str:
    if not path.exists():
        return default
    return path.read_text(encoding="utf-8").strip()

SCHEMA_SQL_TEXT = read_text(SCHEMA_SQL_PATH)
SCHEMA_GUIDE_TEXT = read_text(SCHEMA_GUIDE_PATH)
BUSINESS_NARRATIVE_TEXT = read_text(BUSINESS_NARRATIVE_PATH)

print("schema_sql chars:", len(SCHEMA_SQL_TEXT))
print("schema_guide chars:", len(SCHEMA_GUIDE_TEXT))
print("business_narrative chars:", len(BUSINESS_NARRATIVE_TEXT))


In [ ]:
# Load judge outputs from by_model
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    if not path.exists():
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_model_rows(alias: str, source_kind: str = "pass") -> pd.DataFrame:
    model_dir = BY_MODEL_DIR / alias
    pass_path = model_dir / "ttsql_candidates_judge_pass.jsonl"
    judged_path = model_dir / "ttsql_candidates_judged.jsonl"

    if source_kind == "pass":
        rows = load_jsonl(pass_path)
    elif source_kind == "judged":
        rows = load_jsonl(judged_path)
        rows = [r for r in rows if bool(r.get("judge_pass", False))]
    else:
        raise ValueError(f"Unknown source_kind={source_kind}")

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Safety filter if judged file is loaded directly.
    if "judge_score" in df.columns:
        df = df[(df["judge_score"].fillna(0) >= PASS_SCORE_THRESHOLD)]
    if "judge_verdict" in df.columns:
        df = df[df["judge_verdict"].fillna("") == "pass"]

    df["source_model_alias"] = alias
    return df


frames = []
for alias in MODEL_ALIASES:
    df_alias = load_model_rows(alias, source_kind=SOURCE_KIND)
    print(f"{alias}: {len(df_alias)} rows")
    if not df_alias.empty:
        frames.append(df_alias)

if not frames:
    raise RuntimeError("No rows loaded from by_model outputs.")

raw_df = pd.concat(frames, ignore_index=True)
print("Total loaded rows:", len(raw_df))
raw_df.head(3)


In [ ]:
# Clean + deduplicate
df = raw_df.copy()

required_cols = ["question_fr", "sql"]
for col in required_cols:
    if col not in df.columns:
        raise KeyError(f"Missing required column: {col}")

df = df.dropna(subset=required_cols).copy()
df["question_fr"] = df["question_fr"].astype(str).str.strip()
df["sql"] = df["sql"].astype(str).str.strip()
df = df[(df["question_fr"] != "") & (df["sql"] != "")]

# Keep one sample per (question, sql) pair.
df = df.drop_duplicates(subset=["question_fr", "sql"]).reset_index(drop=True)

if "difficulty" not in df.columns:
    df["difficulty"] = "unknown"

print("Rows after cleaning/dedup:", len(df))
print("Rows by source model:")
print(df.groupby("source_model_alias").size().sort_values(ascending=False))
print("\nRows by difficulty:")
print(df.groupby("difficulty").size().sort_values(ascending=False))


In [ ]:
# Build training prompts and HF splits
PROMPT_CONTEXT = (
    "### SFIL Business Context\n"
    + BUSINESS_NARRATIVE_TEXT[:12000]
    + "\n\n### SQLite Schema (DDL)\n"
    + SCHEMA_SQL_TEXT[:12000]
    + "\n\n### Schema Guide Excerpt\n"
    + SCHEMA_GUIDE_TEXT[:12000]
)


def build_prompt(question: str) -> str:
    system_message = (
        "You are an expert Text-to-SQL assistant for SFIL municipal finance analysis.\n"
        "Return ONLY one SQLite SELECT query, with no explanation and no markdown.\n"
        "Rules:\n"
        "- Use only schema elements from the provided context.\n"
        "- Read-only SQL only (SELECT / WITH ... SELECT).\n"
        "- No INSERT/UPDATE/DELETE/DDL/PRAGMA.\n"
        + PROMPT_CONTEXT
    )
    return (
        f"<|im_start|>system\n{system_message}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


df_ft = df.copy()
df_ft["prompt"] = df_ft["question_fr"].apply(build_prompt)
df_ft["answer"] = df_ft["sql"]

full_dataset = Dataset.from_pandas(
    df_ft[["question_fr", "answer", "prompt", "source_model_alias", "difficulty"]],
    preserve_index=False,
)

train_split = full_dataset.train_test_split(test_size=0.2, seed=SEED)
train_dataset = train_split["train"]
rest_split = train_split["test"].train_test_split(test_size=0.5, seed=SEED)
val_dataset = rest_split["train"]
test_dataset = rest_split["test"]

dataset = DatasetDict(train=train_dataset, validation=val_dataset, test=test_dataset)
print({k: len(v) for k, v in dataset.items()})


def format_prompt_train(examples):
    return [p + a.strip() + "\n<|im_end|>" for p, a in zip(examples["prompt"], examples["answer"])]


def format_prompt_eval(example):
    return example["prompt"]


## Optional SFT Training (Unsloth + LoRA)

This section is intentionally disabled by default (`ENABLE_TRAINING = False`).
Enable it only when your GPU environment and dependencies are ready.


In [ ]:
# Optional training configuration
ENABLE_TRAINING = False

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True

PER_DEVICE_BATCH_SIZE = 2
GRAD_ACC_STEPS = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 2

ADAPTER_OUTPUT_DIR = OUTPUT_DIR / "qwen_sft_adapter"
RUN_OUTPUT_DIR = OUTPUT_DIR / "runs" / "qwen_sft"

print("Training enabled:", ENABLE_TRAINING)
print("Model:", MODEL_ID)
print("Adapter output:", ADAPTER_OUTPUT_DIR)


In [ ]:
# Optional training execution
if ENABLE_TRAINING:
    import gc
    import torch
    from transformers import TrainingArguments
    from trl import SFTTrainer
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=torch.bfloat16,
        load_in_4bit=LOAD_IN_4BIT,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=64,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    training_args = TrainingArguments(
        output_dir=str(RUN_OUTPUT_DIR),
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        logging_steps=20,
        save_strategy="epoch",
        bf16=True,
        fp16=False,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        formatting_func=format_prompt_train,
        args=training_args,
        packing=True,
        max_seq_length=MAX_SEQ_LENGTH,
    )

    gc.collect()
    torch.cuda.empty_cache()

    print("Starting SFT training...")
    trainer.train()
    trainer.save_model(str(ADAPTER_OUTPUT_DIR))
    print("Adapter saved to:", ADAPTER_OUTPUT_DIR)
else:
    print("Training skipped. Set ENABLE_TRAINING = True to run SFT.")


In [ ]:
# Export prepared splits for reproducibility / handoff
EXPORT_DIR = OUTPUT_DIR / "prepared"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, ds in dataset.items():
    split_df = ds.to_pandas()

    jsonl_path = EXPORT_DIR / f"sft_{split_name}.jsonl"
    csv_path = EXPORT_DIR / f"sft_{split_name}.csv"

    split_df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
    split_df.to_csv(csv_path, index=False)

    print(f"[{split_name}] rows={len(split_df)} -> {jsonl_path.name}, {csv_path.name}")
